# 회귀 실습 — Diamonds 가격 예측

이번엔 앞서 연습한 EDA를 바탕으로 실제 **회귀 문제**를 하나 풀어보겠습니다.

사용할 데이터는 다이아몬드의 특성과 가격이 담긴 **Diamonds** 데이터셋입니다. seaborn에 내장되어 있어 별도 다운로드 없이 바로 불러올 수 있습니다.

목표는 캐럿, 컷, 색상, 투명도 등 다이아몬드의 특성으로 **가격(`price`)을 예측**하는 것입니다.

### 데이터 기본 정보

| 컬럼 | 뜻 |
|---|---|
| `carat` | 캐럿 (무게) |
| `cut` | 컷 등급 (Fair < Good < Very Good < Premium < Ideal) |
| `color` | 색상 등급 (J가 가장 낮고 D가 가장 높음) |
| `clarity` | 투명도 등급 (I1이 가장 낮고 IF가 가장 높음) |
| `depth` | 깊이 비율 (%) |
| `table` | 테이블 비율 (%) |
| `price` | **가격 (정답, 단위: 달러)** |
| `x`, `y`, `z` | 가로/세로/깊이 길이 (mm) |

`cut`, `color`, `clarity`는 순서가 있는 범주형(순서형) 변수라는 점이 특징입니다. 하나씩 살펴보겠습니다.

## 1. 데이터 불러오기 & 기본 정보

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("diamonds")
df.head()

In [ ]:
print(df.shape)
df.info()

약 5만 4천 개의 관측치가 있고, `cut`/`color`/`clarity`가 문자열(object, 정확히는 category)입니다.

결측치도 확인해보겠습니다.

In [ ]:
df.isna().sum()

결측치는 없네요. 이번엔 예측 대상인 `price`의 분포부터 확인해보겠습니다.

## 2. 타겟(price) 분포 확인

In [ ]:
df['price'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(df['price'], bins=50)
axes[0].set_title("price")

axes[1].hist(np.log1p(df['price']), bins=50)
axes[1].set_title("log(price)")
plt.tight_layout()
plt.show()

`price`는 저가 다이아몬드 쪽에 크게 몰려있고 고가로 갈수록 꼬리가 길게 늘어지는 분포입니다. 이렇게 한쪽으로 치우친(skewed) 타겟은 로그 변환을 하면 훨씬 정규분포에 가까워지는 걸 볼 수 있습니다.

이 점은 나중에 모델을 학습할 때 다시 활용해보겠습니다. 일단은 원래 `price`로 진행합니다.

## 3. 특성 살펴보기

먼저 수치형 변수들이 `price`와 어떤 관계가 있는지 산점도로 확인해보겠습니다.

In [ ]:
num_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, col in zip(axes.flatten(), num_cols):
    ax.scatter(df[col], df['price'], alpha=0.05, s=5)
    ax.set_title(f"{col} vs price")
plt.tight_layout()
plt.show()

`carat`이 `price`와 가장 뚜렷한 관계를 보입니다. 캐럿이 커질수록 가격이 올라가는데, 일직선이라기보단 위로 휘어지는 형태입니다 (역시 로그 변환이 도움될 수 있는 부분입니다).

`x`, `y`, `z`(크기)도 `carat`과 비슷한 패턴을 보이는데, 사실 캐럿 자체가 부피(크기)와 밀접한 관련이 있어서 그렇습니다. `depth`, `table`은 상대적으로 `price`와의 관계가 뚜렷하지 않네요.

이번엔 범주형 변수(`cut`, `color`, `clarity`)별로 가격 분포가 어떻게 다른지 박스플롯으로 확인해보겠습니다.

In [ ]:
cat_cols = ['cut', 'color', 'clarity']
cat_orders = {
    'cut': ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'],
    'color': ['J', 'I', 'H', 'G', 'F', 'E', 'D'],
    'clarity': ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF'],
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, cat_cols):
    sns.boxplot(data=df, x=col, y='price', order=cat_orders[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

등급이 좋을수록(오른쪽으로 갈수록) 가격이 비쌀 것 같지만, 실제론 그렇게 뚜렷하지 않죠? 오히려 `cut`이 Ideal인데도 가격 중앙값이 낮아 보입니다.

왜 이런 결과가 나올까요? 힌트는 `carat`에 있습니다. 등급이 좋은 다이아몬드라도 캐럿(크기) 자체가 작으면 가격이 낮을 수 있기 때문에, 등급의 효과가 캐럿에 가려져서 잘 안 보이는 것입니다. 이런 경우 `carat` 대비 가격이 어떻게 다른지를 봐야 좀 더 정확한 비교가 됩니다. 지금은 일단 참고만 하고 넘어가겠습니다.

## 4. 범주형 인코딩

`cut`, `color`, `clarity`는 순서가 있는 범주형(순서형) 변수이므로, 순서 정보를 살리는 순서형 인코딩을 적용하겠습니다.

In [ ]:
X = df.drop(columns=['price'])
y = df['price']

for col in cat_cols:
    order = cat_orders[col]
    mapping = {v: i for i, v in enumerate(order)}
    X[col] = X[col].map(mapping)

X.head()

등급이 낮을수록 0, 높을수록 큰 숫자로 바뀐 걸 확인할 수 있습니다.

## 5. 평가지표 정하기 & train/test 분리

가격을 예측하는 문제이므로 예측값과 실제값의 차이(오차)를 그대로 보여주는 **MAE(평균 절대 오차)**를 기본 지표로 사용하겠습니다. 실제 달러 단위로 해석하기도 쉽습니다.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 6. 베이스라인 모델 학습

먼저 선형회귀로 베이스라인을 만들어보겠습니다.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
print(f"MAE: {mae:.1f}")
print(f"RMSE: {rmse:.1f}")

평균적으로 실제 가격과 500달러 안팎의 차이가 나네요. `price`의 평균이 약 3,900달러였던 걸 감안하면 꽤 큰 오차입니다.

RMSE가 MAE보다 훨씬 크다는 것도 눈여겨볼 부분입니다. 이는 오차가 클수록 제곱으로 커지는 RMSE 특성상, **일부 데이터에서 유독 큰 오차**가 나고 있다는 신호입니다. 어디서 이런 오차가 나는지 살펴보겠습니다.

## 7. 오차 분석

예측이 크게 틀린 구간이 있는지, `carat`(캐럿) 기준으로 나눠서 살펴보겠습니다.

In [ ]:
result = X_test.copy()
result['price'] = y_test
result['pred'] = pred
result['error'] = result['pred'] - result['price']
result['abs_error'] = result['error'].abs()

result.sort_values('abs_error', ascending=False).head(10)

가장 오차가 큰 케이스들을 보면 캐럿이 큰(고가) 다이아몬드에 몰려있는 걸 볼 수 있습니다. 캐럿 구간별로 평균 오차를 집계해서 더 명확히 확인해보겠습니다.

In [ ]:
result['carat_bin'] = pd.cut(result['carat'], bins=[0, 0.5, 1, 1.5, 2, 5])
result.groupby('carat_bin')[['abs_error']].mean()

역시 캐럿이 커질수록(비싼 다이아몬드일수록) 평균 절대 오차도 같이 커집니다. 저가 구간에서는 몇십 달러 수준이지만 고가 구간에서는 몇백~천 달러 단위로 틀리고 있습니다.

이건 앞서 봤던 `price`의 치우친 분포와 관련이 있습니다. 선형회귀는 절대 오차(차이)를 기준으로 맞추다 보니, 가격이 큰 쪽에서 상대적으로 더 크게 틀려도 학습 과정에서 잘 드러나지 않을 수 있습니다.

## 8. 개선 시도 — 타겟 로그 변환

`price`를 그대로 예측하는 대신, `log(price)`를 예측하도록 바꿔서 같은 방식으로 다시 학습해보겠습니다.

In [ ]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

model_log = LinearRegression()
model_log.fit(X_train, y_train_log)

pred_log = model_log.predict(X_test)
pred_from_log = np.expm1(pred_log)   # 다시 원래 가격 단위로 복원

mae_log = mean_absolute_error(y_test, pred_from_log)
rmse_log = mean_squared_error(y_test, pred_from_log) ** 0.5
print(f"MAE (log 변환 후 복원): {mae_log:.1f}")
print(f"RMSE (log 변환 후 복원): {rmse_log:.1f}")

처음 베이스라인과 비교해보면 MAE와 RMSE 모두 개선된 걸 볼 수 있습니다.

`log` 변환을 하면 모델이 절대적인 달러 차이 대신 **상대적인(비율) 차이**를 기준으로 학습하게 되어서, 고가 구간에서의 큰 오차에 덜 휘둘리게 됩니다. 캐럿 구간별로 다시 확인해보겠습니다.

In [ ]:
result['pred_log'] = pred_from_log
result['abs_error_log'] = (result['pred_log'] - result['price']).abs()

result.groupby('carat_bin')[['abs_error', 'abs_error_log']].mean()

고가 구간에서의 오차가 확실히 줄어든 걸 볼 수 있습니다. 대신 저가 구간에서는 큰 차이가 없거나 오히려 미세하게 늘 수도 있는데, 이는 트레이드오프로 이해하면 됩니다.

## 9. 정리

- `carat`(캐럿)이 `price`를 설명하는 가장 강력한 변수였고, `cut`/`color`/`clarity` 같은 등급 변수의 효과는 캐럿에 가려져 단순 비교로는 잘 드러나지 않았습니다.
- `price`가 한쪽으로 치우친 분포를 가지고 있어서, 그대로 예측하면 고가 구간에서 오차가 크게 몰리는 문제가 있었습니다.
- 타겟을 로그 변환해서 학습하니 특히 고가 구간의 오차가 줄어들며 전체 지표도 개선됐습니다.

**생각해볼 질문:** 지금은 선형회귀만 써봤는데, 만약 트리 기반 모델(예: RandomForest)을 쓴다면 로그 변환 없이도 비슷한 효과를 볼 수 있을까요? 그리고 `carat`과 `x`/`y`/`z`처럼 서로 강하게 연관된 변수들을 그대로 다 넣는 게 항상 좋은 선택일까요?